<a href="https://colab.research.google.com/github/AbouDjafar/5G-Safe-DQN/blob/main/memoire_5g_safe_dqn_graphe_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Memoire Master II IRS UMa - Safe DQN guide par graphe

Ce notebook est prepare pour Google Colab. Il monte Google Drive, telecharge le dataset depuis le cloud si necessaire, entraine le Safe DQN guide par graphe, puis sauvegarde les resultats dans Drive.


## 1. Monter Google Drive

Executez cette cellule en premier. Colab demandera une autorisation pour acceder a votre Drive. Les donnees, figures et CSV seront stockes dans `MyDrive/Implementation_Lahibe`.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


## 2. Imports, chemins et parametres globaux

Cette cellule charge les bibliotheques et fixe les parametres de l'experience : nombre de stations, actions, seuils, chemins Drive et URL du dataset.


In [2]:
# -*- coding: utf-8 -*-
"""Safe DQN guide par graphe pour l'economie d'energie 5G.

Ce script genere les resultats et les figures utilises dans le memoire.
Il est volontairement autonome et n'utilise que numpy, pandas et matplotlib
pour pouvoir fonctionner meme lorsque PyTorch n'est pas disponible localement.
Le notebook Colab genere dans le meme dossier reprend la meme logique avec
une structure par cellules.
"""
import json
import math
import random
import shutil
import urllib.request
from collections import deque
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_URL = "https://raw.githubusercontent.com/TechLeoo/Predictive-Modeling-for-5G-Energy-Consumption/main/5G_energy_consumption_dataset.csv"
N_CELLS = 6
SCALABILITY_CELLS = [6, 16, 32, 64]
TOP_K_NEIGHBORS = 3
CORR_THRESHOLD = 0.45
TRAIN_RATIO = 0.70
ACTIONS = {0: "Actif", 1: "Economie partielle", 2: "Veille"}
ENERGY_MULT = np.array([1.00, 0.65, 0.18], dtype=float)
CAPACITY = np.array([1.00, 0.72, 0.10], dtype=float)
REWARD_WEIGHTS = {
    "energy": 1.20,
    "blocking": 5.00,
    "neighbor": 1.30,
    "switch": 0.06,
    "invalid": 3.00,
}

# Google Colab / Google Drive paths.
WORK_DIR = Path("/content/drive/MyDrive/Implementation_Lahibe")
DATA_DIR = WORK_DIR / "donnees_5g_energy_itu_zindi"
RESULTS_DIR = WORK_DIR / "results_safe_dqn_graphe"
EXPORT_DIR = WORK_DIR / "exports_for_latex"
LATEX_RESULTS_DIR = EXPORT_DIR / "results_rl"
LATEX_MANUAL_DIR = EXPORT_DIR / "manual"
CSV_PATH = DATA_DIR / "5G_energy_consumption_dataset.csv"


## 3. Creation des dossiers et chargement du dataset

Le dataset est pris depuis le cloud via l'URL GitHub. S'il existe deja dans Drive, il n'est pas retelecharge.


In [3]:
def ensure_dirs() -> None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    LATEX_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    LATEX_MANUAL_DIR.mkdir(parents=True, exist_ok=True)


def load_data() -> pd.DataFrame:
    if not CSV_PATH.exists():
        urllib.request.urlretrieve(DATA_URL, CSV_PATH)
    df = pd.read_csv(CSV_PATH)
    expected = {"Time", "BS", "Energy", "load", "ESMODE", "TXpower"}
    missing = expected.difference(df.columns)
    if missing:
        raise ValueError(f"Colonnes manquantes: {sorted(missing)}")
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["Time"], format="%Y%m%d %H%M%S")
    df = df.rename(columns={"BS": "cell_id", "Energy": "energy", "TXpower": "tx_power", "ESMODE": "esmode"})
    df = df[["timestamp", "cell_id", "energy", "load", "esmode", "tx_power"]]
    df = df.dropna().sort_values(["timestamp", "cell_id"]).reset_index(drop=True)
    for col in ["energy", "load", "esmode", "tx_power"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna()
    df["load"] = df["load"].clip(0, 1)
    return df


def select_cells(df: pd.DataFrame, n_cells: int = N_CELLS) -> pd.DataFrame:
    counts = df.groupby("cell_id")["timestamp"].nunique().sort_values(ascending=False)
    candidates = counts.head(max(n_cells * 4, n_cells)).index.tolist()
    stats = df[df["cell_id"].isin(candidates)].groupby("cell_id").agg(
        n_obs=("timestamp", "nunique"),
        load_mean=("load", "mean"),
        load_std=("load", "std"),
        energy_mean=("energy", "mean"),
    )
    stats["score"] = (
        stats["n_obs"] / max(stats["n_obs"].max(), 1e-9)
        + stats["load_mean"]
        + stats["load_std"].fillna(0)
        + stats["energy_mean"] / max(stats["energy_mean"].max(), 1e-9)
    )
    cells = stats.sort_values("score", ascending=False).head(n_cells).index.tolist()
    out = df[df["cell_id"].isin(cells)].copy()
    out["cell_idx"] = out["cell_id"].astype("category").cat.codes
    return out.sort_values(["timestamp", "cell_id"]).reset_index(drop=True)


## 4. Graphe de correlation et variables d'etat

Ces fonctions construisent le graphe entre stations, puis ajoutent les variables observees par l'agent : charge, energie, heure, historique court et pression des voisines.


In [4]:
def build_graph(df: pd.DataFrame):
    pivot = df.pivot_table(index="timestamp", columns="cell_id", values="load", aggfunc="mean").sort_index()
    corr = pivot.corr().fillna(0.0)
    cells = list(corr.columns)
    neighbors = {}
    edges = []
    for cell in cells:
        s = corr[cell].drop(cell).abs().sort_values(ascending=False)
        chosen = s.head(TOP_K_NEIGHBORS)
        neighbors[cell] = chosen.index.tolist()
        for nb, weight in chosen.items():
            if weight >= CORR_THRESHOLD:
                edge = tuple(sorted((cell, nb)))
                edges.append((edge[0], edge[1], float(weight)))
    unique = {}
    for a, b, w in edges:
        unique[(a, b)] = max(unique.get((a, b), 0), w)
    edge_df = pd.DataFrame([{"source": a, "target": b, "weight": w} for (a, b), w in unique.items()])
    if edge_df.empty:
        for cell in cells:
            s = corr[cell].drop(cell).abs().sort_values(ascending=False).head(1)
            for nb, weight in s.items():
                edge = tuple(sorted((cell, nb)))
                unique[edge] = max(unique.get(edge, 0), float(weight))
        edge_df = pd.DataFrame([{"source": a, "target": b, "weight": w} for (a, b), w in unique.items()])
    return corr, neighbors, edge_df


def build_complete_graph(df: pd.DataFrame, selected_cells: list[str], selected_edge_df: pd.DataFrame):
    pivot = df.pivot_table(index="timestamp", columns="cell_id", values="load", aggfunc="mean").sort_index()
    corr = pivot.corr().fillna(0.0)
    cells = list(corr.columns)
    selected_pairs = {tuple(sorted((r.source, r.target))) for r in selected_edge_df.itertuples(index=False)}
    records = {}
    for source in cells:
        ranked = corr[source].drop(source).abs().sort_values(ascending=False)
        if ranked.empty:
            continue
        target = ranked.index[0]
        edge = tuple(sorted((source, target)))
        records[edge] = max(records.get(edge, 0.0), float(ranked.iloc[0]))
    for source, target in selected_pairs:
        edge = tuple(sorted((source, target)))
        records[edge] = max(records.get(edge, 0.0), float(abs(corr.loc[source, target])))
    edge_df = pd.DataFrame([
        {
            "source": a,
            "target": b,
            "weight": w,
            "selected_source": a in selected_cells,
            "selected_target": b in selected_cells,
            "selected_edge": tuple(sorted((a, b))) in selected_pairs,
        }
        for (a, b), w in records.items()
    ])
    return corr, edge_df


def add_features(df: pd.DataFrame, neighbors: dict[str, list[str]]) -> pd.DataFrame:
    d = df.copy()
    max_energy = max(d["energy"].max(), 1e-9)
    max_tx = max(d["tx_power"].max(), 1e-9)
    max_es = max(d["esmode"].max(), 1e-9)
    d["energy_norm"] = d["energy"] / max_energy
    d["tx_norm"] = d["tx_power"] / max_tx
    d["esmode_norm"] = d["esmode"] / max_es
    d["hour"] = d["timestamp"].dt.hour
    d["hour_sin"] = np.sin(2 * np.pi * d["hour"] / 24.0)
    d["hour_cos"] = np.cos(2 * np.pi * d["hour"] / 24.0)
    d["prev_load"] = d.groupby("cell_id")["load"].shift(1).fillna(d["load"])
    d["roll3_load"] = d.groupby("cell_id")["load"].rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
    load_lookup = d.pivot_table(index="timestamp", columns="cell_id", values="load", aggfunc="mean")
    neigh_mean = []
    neigh_pressure = []
    for row in d[["timestamp", "cell_id"]].itertuples(index=False):
        nbs = neighbors.get(row.cell_id, [])
        vals = []
        for nb in nbs:
            try:
                vals.append(float(load_lookup.loc[row.timestamp, nb]))
            except Exception:
                pass
        if vals:
            m = float(np.mean(vals))
            p = float(np.max(vals))
        else:
            m = float(d.loc[(d["timestamp"] == row.timestamp) & (d["cell_id"] != row.cell_id), "load"].mean())
            p = m
        neigh_mean.append(m if not math.isnan(m) else 0.0)
        neigh_pressure.append(p if not math.isnan(p) else 0.0)
    d["neighbor_load_mean"] = neigh_mean
    d["neighbor_pressure"] = neigh_pressure
    d["load_delta"] = d.groupby("cell_id")["load"].diff().fillna(0).abs()
    return d.reset_index(drop=True)


FULL_FEATURES = ["load", "energy_norm", "tx_norm", "esmode_norm", "hour_sin", "hour_cos", "prev_load", "roll3_load", "neighbor_load_mean", "neighbor_pressure"]
NO_GRAPH_FEATURES = ["load", "energy_norm", "tx_norm", "esmode_norm", "hour_sin", "hour_cos", "prev_load", "roll3_load"]


## 5. Actions, masque QoS et recompense

Cette partie definit les actions possibles, interdit les actions dangereuses avec le masque QoS, puis calcule l'effet energetique et le trafic bloque.


In [5]:
def valid_actions(row) -> list[int]:
    valid = [0]
    if row.load <= 0.85 and row.neighbor_pressure <= 0.90:
        valid.append(1)
    if row.load <= 0.28 and row.roll3_load <= 0.32 and row.neighbor_load_mean <= 0.58 and row.neighbor_pressure <= 0.72:
        valid.append(2)
    return valid


def action_effect(row, action: int, prev_action: int | None, enforce_mask: bool) -> dict:
    invalid = 0.0
    if enforce_mask and action not in valid_actions(row):
        invalid = 1.0
        action = 1 if 1 in valid_actions(row) else 0
    multiplier = ENERGY_MULT[action]
    energy = float(row.energy) * multiplier
    neighbor_relief = max(0.0, min(0.18, 0.65 - float(row.neighbor_load_mean)))
    effective_capacity = CAPACITY[action]
    if action == 2:
        effective_capacity += neighbor_relief
    elif action == 1:
        effective_capacity += 0.08 * neighbor_relief
    demand = max(float(row.load), 1e-6)
    served = min(demand, effective_capacity)
    blocked = max(0.0, demand - effective_capacity)
    blocking_rate = blocked / demand
    neighbor_penalty = max(0.0, float(row.neighbor_load_mean) + 0.35 * blocked - 0.75)
    switch = 0.0 if prev_action is None or prev_action == action else 1.0
    reward = (
        REWARD_WEIGHTS["energy"] * (1.0 - multiplier)
        - REWARD_WEIGHTS["blocking"] * blocking_rate
        - REWARD_WEIGHTS["neighbor"] * neighbor_penalty
        - REWARD_WEIGHTS["switch"] * switch
        - REWARD_WEIGHTS["invalid"] * invalid
    )
    return {
        "action": action,
        "energy": energy,
        "demand": demand,
        "served": served,
        "blocked": blocked,
        "blocking_rate": blocking_rate,
        "switch": switch,
        "invalid": invalid,
        "reward": reward,
    }


## 6. Definition du DQN

Le DQN estime la valeur de chaque action pour un etat donne. Cette version est volontairement implementee avec NumPy pour rester legere et lisible.


In [6]:
@dataclass
class Transition:
    s: np.ndarray
    a: int
    r: float
    ns: np.ndarray
    done: bool
    next_valid: list[int]


class NumpyDQN:
    def __init__(self, n_features: int, n_actions: int = 3, hidden: int = 32, lr: float = 0.0015, gamma: float = 0.92):
        self.n_actions = n_actions
        self.lr = lr
        self.gamma = gamma
        scale1 = math.sqrt(2.0 / n_features)
        scale2 = math.sqrt(2.0 / hidden)
        self.W1 = np.random.randn(n_features, hidden) * scale1
        self.b1 = np.zeros(hidden)
        self.W2 = np.random.randn(hidden, n_actions) * scale2
        self.b2 = np.zeros(n_actions)
        self.target = self.copy_params()

    def copy_params(self):
        return [self.W1.copy(), self.b1.copy(), self.W2.copy(), self.b2.copy()]

    def sync_target(self):
        self.target = self.copy_params()

    def forward(self, x: np.ndarray, target: bool = False):
        W1, b1, W2, b2 = self.target if target else (self.W1, self.b1, self.W2, self.b2)
        z1 = x @ W1 + b1
        h = np.maximum(0, z1)
        q = h @ W2 + b2
        return q, z1, h

    def predict(self, x: np.ndarray, valid: list[int] | None = None) -> int:
        q, _, _ = self.forward(x)
        if valid is not None:
            masked = np.full_like(q, -1e9)
            masked[valid] = q[valid]
            return int(np.argmax(masked))
        return int(np.argmax(q))

    def train_batch(self, batch: list[Transition]) -> float:
        losses = []
        for tr in batch:
            q, z1, h = self.forward(tr.s)
            q_target_next, _, _ = self.forward(tr.ns, target=True)
            if tr.done:
                y = tr.r
            else:
                if tr.next_valid:
                    y = tr.r + self.gamma * float(np.max(q_target_next[tr.next_valid]))
                else:
                    y = tr.r + self.gamma * float(np.max(q_target_next))
            err = q[tr.a] - y
            losses.append(float(err * err))
            dq = np.zeros(self.n_actions)
            dq[tr.a] = 2.0 * err
            dW2 = np.outer(h, dq)
            db2 = dq
            dh = self.W2 @ dq
            dz1 = dh * (z1 > 0)
            dW1 = np.outer(tr.s, dz1)
            db1 = dz1
            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2
            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1
        return float(np.mean(losses)) if losses else 0.0


## 7. Entrainement et evaluation

Ces fonctions entrainent le DQN, separent train/test, comparent les politiques et calculent les indicateurs energie-QoS.


In [7]:
def prepare_arrays(df: pd.DataFrame, features: list[str]):
    X = df[features].to_numpy(dtype=float)
    mu = X.mean(axis=0)
    sigma = X.std(axis=0) + 1e-9
    return (X - mu) / sigma, mu, sigma


def train_dqn(train_df: pd.DataFrame, features: list[str], safe: bool, episodes: int = 55,
              train_stride: int = 5, batch_size: int = 32):
    X, _, _ = prepare_arrays(train_df, features)
    model = NumpyDQN(n_features=len(features))
    buffer: deque[Transition] = deque(maxlen=6000)
    rewards_hist = []
    loss_hist = []
    prev_actions = {c: 0 for c in train_df["cell_id"].unique()}
    rows = list(train_df.itertuples(index=False))
    eps_start, eps_end = 0.85, 0.08
    for ep in range(episodes):
        eps = eps_end + (eps_start - eps_end) * math.exp(-ep / 18.0)
        total_reward = 0.0
        order = np.arange(len(rows) - 1)
        if ep % 2 == 1:
            np.random.shuffle(order)
        for idx in order:
            row = rows[idx]
            next_row = rows[idx + 1]
            valid = valid_actions(row) if safe else [0, 1, 2]
            if random.random() < eps:
                action = int(random.choice(valid))
            else:
                action = model.predict(X[idx], valid)
            prev = prev_actions.get(row.cell_id, 0)
            eff = action_effect(row, action, prev, enforce_mask=safe)
            prev_actions[row.cell_id] = eff["action"]
            done = bool(idx == len(rows) - 2)
            next_valid = valid_actions(next_row) if safe else [0, 1, 2]
            buffer.append(Transition(X[idx], eff["action"], eff["reward"], X[idx + 1], done, next_valid))
            total_reward += eff["reward"]
            if len(buffer) >= batch_size and idx % train_stride == 0:
                batch = random.sample(buffer, batch_size)
                loss_hist.append(model.train_batch(batch))
        if ep % 5 == 0:
            model.sync_target()
        rewards_hist.append(total_reward / max(len(order), 1))
    return model, np.array(rewards_hist), np.array(loss_hist)


def split_train_test(df_feat: pd.DataFrame):
    timestamps = sorted(df_feat["timestamp"].unique())
    split_ts = timestamps[int(len(timestamps) * TRAIN_RATIO)]
    train_df = df_feat[df_feat["timestamp"] < split_ts].reset_index(drop=True)
    test_df = df_feat[df_feat["timestamp"] >= split_ts].reset_index(drop=True)
    return train_df, test_df


def evaluate_all_policies(test_df: pd.DataFrame, safe_model: NumpyDQN, raw_model: NumpyDQN):
    evals = {}
    metrics = []
    for key, model, features, safe in [
        ("always_on", None, FULL_FEATURES, False),
        ("static", None, FULL_FEATURES, False),
        ("static_graph_safe", None, FULL_FEATURES, True),
        ("dqn", raw_model, NO_GRAPH_FEATURES, False),
        ("safe_dqn_graph", safe_model, FULL_FEATURES, True),
    ]:
        out, met = evaluate_policy(test_df, features, model, key, safe=safe)
        evals[key] = out
        metrics.append(met)
    return evals, pd.DataFrame(metrics)


def run_experiment(raw_df: pd.DataFrame, n_cells: int, safe_episodes: int, raw_episodes: int,
                   train_stride: int = 5, batch_size: int = 32):
    df = select_cells(raw_df, n_cells=n_cells)
    corr, neighbors, edge_df = build_graph(df)
    df_feat = add_features(df, neighbors)
    train_df, test_df = split_train_test(df_feat)
    safe_model, rewards, losses = train_dqn(
        train_df, FULL_FEATURES, safe=True, episodes=safe_episodes,
        train_stride=train_stride, batch_size=batch_size
    )
    raw_model, _, _ = train_dqn(
        train_df, NO_GRAPH_FEATURES, safe=False, episodes=raw_episodes,
        train_stride=train_stride, batch_size=batch_size
    )
    evals, summary = evaluate_all_policies(test_df, safe_model, raw_model)
    return {
        "n_cells": n_cells,
        "df": df,
        "df_feat": df_feat,
        "corr": corr,
        "neighbors": neighbors,
        "edge_df": edge_df,
        "train_df": train_df,
        "test_df": test_df,
        "safe_model": safe_model,
        "raw_model": raw_model,
        "rewards": rewards,
        "losses": losses,
        "evals": evals,
        "summary": summary,
    }


def evaluate_policy(df: pd.DataFrame, features: list[str], model: NumpyDQN | None, policy: str, safe: bool = False):
    X, _, _ = prepare_arrays(df, features)
    prev_actions = {c: 0 for c in df["cell_id"].unique()}
    records = []
    for i, row in enumerate(df.itertuples(index=False)):
        if policy == "always_on":
            action = 0
            enforce = False
        elif policy == "static":
            action = 2 if row.load < 0.35 else (1 if row.load < 0.55 else 0)
            enforce = False
        elif policy == "static_graph_safe":
            desired = 2 if row.load < 0.35 else (1 if row.load < 0.55 else 0)
            valid = valid_actions(row)
            action = desired if desired in valid else (1 if 1 in valid and desired == 2 else 0)
            enforce = True
        elif policy in ("dqn", "safe_dqn_graph"):
            valid = valid_actions(row) if safe else [0, 1, 2]
            action = model.predict(X[i], valid)
            enforce = safe
        else:
            raise ValueError(policy)
        prev = prev_actions.get(row.cell_id, 0)
        eff = action_effect(row, action, prev, enforce_mask=enforce)
        prev_actions[row.cell_id] = eff["action"]
        rec = {"timestamp": row.timestamp, "cell_id": row.cell_id, "load": row.load, "hour": row.hour, **eff}
        records.append(rec)
    out = pd.DataFrame(records)
    demand = out["demand"].sum()
    active_energy = float(df["energy"].sum())
    metrics = {
        "strategy": policy,
        "energy": out["energy"].sum(),
        "energy_gain_pct": 100.0 * (active_energy - out["energy"].sum()) / max(active_energy, 1e-9),
        "served_pct": 100.0 * out["served"].sum() / max(demand, 1e-9),
        "blocking_pct": 100.0 * out["blocked"].sum() / max(demand, 1e-9),
        "switches_per_100": 100.0 * out["switch"].sum() / max(len(out), 1),
        "invalid_actions_pct": 100.0 * out["invalid"].sum() / max(len(out), 1),
        "avg_reward": out["reward"].mean(),
    }
    return out, metrics


def scenario_name(row) -> str:
    if 0 <= row.hour <= 5:
        return "Faible charge nocturne"
    if 10 <= row.hour <= 18:
        return "Forte charge diurne"
    if 19 <= row.hour <= 23:
        return "Pic de soiree"
    return "Trafic variable"


def scenario_metrics(df_eval: dict[str, pd.DataFrame], test_df: pd.DataFrame) -> pd.DataFrame:
    base_energy_by_time = test_df.copy()
    base_energy_by_time["scenario"] = base_energy_by_time.apply(scenario_name, axis=1)
    active_by_scenario = base_energy_by_time.groupby("scenario")["energy"].sum().to_dict()
    rows = []
    for name, out in df_eval.items():
        o = out.copy()
        o["scenario"] = o.apply(scenario_name, axis=1)
        for scen, g in o.groupby("scenario"):
            demand = g["demand"].sum()
            energy = g["energy"].sum()
            active_energy = active_by_scenario.get(scen, energy)
            rows.append({
                "scenario": scen,
                "strategy": name,
                "energy": energy,
                "energy_gain_pct": 100.0 * (active_energy - energy) / max(active_energy, 1e-9),
                "served_pct": 100.0 * g["served"].sum() / max(demand, 1e-9),
                "blocking_pct": 100.0 * g["blocked"].sum() / max(demand, 1e-9),
            })
    return pd.DataFrame(rows)


## 8. Figures principales

Ces fonctions generent les schemas conceptuels et les figures principales : graphe, courbes d'entrainement, histogrammes, compromis energie-QoS.


In [8]:
def plot_conceptual_diagrams():
    def box(ax, center, size, text, fc="#eef4ff", ec="#34568b", fontsize=10):
        x, y = center
        w, h = size
        patch = FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                               boxstyle="round,pad=0.012,rounding_size=0.018",
                               fc=fc, ec=ec, lw=1.5, zorder=2)
        ax.add_patch(patch)
        ax.text(x, y, text, ha="center", va="center", fontsize=fontsize, zorder=3)
        return {"left": (x - w / 2, y), "right": (x + w / 2, y),
                "top": (x, y + h / 2), "bottom": (x, y - h / 2), "center": center}

    def arrow(ax, start, end, color="#333333", lw=1.5, rad=0.0):
        ax.annotate("", xy=end, xytext=start,
                    arrowprops=dict(arrowstyle="-|>", lw=lw, color=color,
                                    shrinkA=0, shrinkB=0,
                                    connectionstyle=f"arc3,rad={rad}"),
                    zorder=4)

    fig, ax = plt.subplots(figsize=(13.6, 4.8))
    ax.axis("off")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    b1 = box(ax, (0.10, 0.62), (0.16, 0.20), "Dataset ITU/Zindi\nmesures 5G")
    b2 = box(ax, (0.30, 0.62), (0.16, 0.20), "Graphe de\ncorrelation")
    b3 = box(ax, (0.50, 0.62), (0.16, 0.20), "Etat RL\ncharge + voisins")
    b4 = box(ax, (0.70, 0.62), (0.16, 0.20), "Safe DQN\nactions masquees")
    b5 = box(ax, (0.90, 0.62), (0.16, 0.20), "Evaluation\nenergie-QoS")
    for a, b in [(b1, b2), (b2, b3), (b3, b4), (b4, b5)]:
        arrow(ax, a["right"], b["left"], lw=1.6)
    bq = box(ax, (0.70, 0.22), (0.24, 0.18), "Masque QoS\ninterdit les actions risquees", fc="#fff5e6", ec="#b66b00")
    arrow(ax, bq["top"], b4["bottom"], color="#b66b00", lw=1.5)
    fig.tight_layout()
    fig.savefig(LATEX_MANUAL_DIR / "chap3_architecture_safe_dqn.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(12.2, 5.4))
    ax.axis("off")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    e = box(ax, (0.16, 0.70), (0.22, 0.18), "Etat s_t\ncharge, energie, voisins", fc="#f3fbf4", ec="#327a3d")
    ag = box(ax, (0.50, 0.70), (0.20, 0.18), "Agent DQN\nQ(s,a)", fc="#f3fbf4", ec="#327a3d")
    ac = box(ax, (0.84, 0.70), (0.24, 0.18), "Action a_t\nactif / economie / veille", fc="#f3fbf4", ec="#327a3d")
    env = box(ax, (0.68, 0.25), (0.28, 0.18), "Environnement 5G simule\ntransition et trafic servi", fc="#f3fbf4", ec="#327a3d")
    rew = box(ax, (0.32, 0.25), (0.24, 0.18), "Recompense r_t\ngain - penalites QoS", fc="#f3fbf4", ec="#327a3d")
    arrow(ax, e["right"], ag["left"], lw=1.45)
    arrow(ax, ag["right"], ac["left"], lw=1.45)
    arrow(ax, ac["bottom"], env["top"], lw=1.35)
    arrow(ax, env["left"], rew["right"], lw=1.35)
    arrow(ax, rew["top"], e["bottom"], lw=1.35)
    fig.tight_layout()
    fig.savefig(LATEX_MANUAL_DIR / "chap3_mdp_safe_dqn.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11.2, 4.8))
    ax.axis("off")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    c1 = box(ax, (0.15, 0.66), (0.22, 0.18), "Actions candidates\nactif, economie, veille", fc="#eef4ff", ec="#34568b")
    c2 = box(ax, (0.43, 0.66), (0.22, 0.18), "Contraintes QoS\ncharge et voisins", fc="#fff5e6", ec="#b66b00")
    c3 = box(ax, (0.72, 0.66), (0.25, 0.18), "Masque d'actions\nveille interdite si risque", fc="#fff5e6", ec="#b66b00")
    c4 = box(ax, (0.72, 0.25), (0.25, 0.18), "Action executee\nparmi les actions valides", fc="#f3fbf4", ec="#327a3d")
    c5 = box(ax, (0.43, 0.25), (0.24, 0.18), "Penalites\nblocage, surcharge, commutation", fc="#fff0f0", ec="#a23b3b", fontsize=9.4)
    arrow(ax, c1["right"], c2["left"], lw=1.45)
    arrow(ax, c2["right"], c3["left"], lw=1.45)
    arrow(ax, c3["bottom"], c4["top"], lw=1.45)
    arrow(ax, c4["left"], c5["right"], lw=1.45)
    fig.tight_layout()
    fig.savefig(LATEX_MANUAL_DIR / "chap3_action_masking_qos.png", dpi=220, bbox_inches="tight")
    plt.close(fig)


def plot_results(df: pd.DataFrame, corr: pd.DataFrame, edge_df: pd.DataFrame,
                 full_corr: pd.DataFrame, full_edge_df: pd.DataFrame,
                 rewards, losses, evals, summary: pd.DataFrame, scen: pd.DataFrame):
    plt.style.use("default")
    colors = {
        "Toujours active": "#d6e9f8",
        "Regle statique": "#9ecae1",
        "Regle statique securisee": "#6baed6",
        "DQN sans graphe": "#2171b5",
        "Safe DQN guide par graphe": "#d62728",
    }

    fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(15.8, 8.0), gridspec_kw={"width_ratios": [1.32, 1.0]})
    selected_cells = list(corr.columns)
    full_cells = list(full_corr.columns)
    n_full = len(full_cells)

    def numeric_key(cell):
        try:
            return int(str(cell).split("_")[-1])
        except Exception:
            return str(cell)

    full_cells = sorted(full_cells, key=numeric_key)
    pos_full = {}
    ring_counts = [160, 220, 260, max(0, n_full - 640)]
    ring_radii = [0.58, 0.82, 1.06, 1.30]
    idx = 0
    for ring_id, count in enumerate(ring_counts):
        if count <= 0:
            continue
        remaining = n_full - idx
        count = min(count, remaining)
        angles = np.linspace(0, 2*np.pi, count, endpoint=False) + ring_id * 0.19
        for angle in angles:
            if idx >= n_full:
                break
            pos_full[full_cells[idx]] = (ring_radii[ring_id] * np.cos(angle), ring_radii[ring_id] * np.sin(angle))
            idx += 1
    if idx < n_full:
        remaining = n_full - idx
        angles = np.linspace(0, 2*np.pi, remaining, endpoint=False) + 0.41
        radius = 1.52
        for angle in angles:
            pos_full[full_cells[idx]] = (radius * np.cos(angle), radius * np.sin(angle))
            idx += 1

    context_edges = full_edge_df[~full_edge_df["selected_edge"]].copy()
    for _, r in context_edges.iterrows():
        if r.source not in pos_full or r.target not in pos_full:
            continue
        x1, y1 = pos_full[r.source]; x2, y2 = pos_full[r.target]
        ax_full.plot([x1, x2], [y1, y2], color="#7aa6c9", lw=0.22 + 0.45 * r.weight, alpha=0.13, zorder=1)
    selected_edges = full_edge_df[full_edge_df["selected_edge"]].copy()
    for _, r in selected_edges.iterrows():
        if r.source not in pos_full or r.target not in pos_full:
            continue
        x1, y1 = pos_full[r.source]; x2, y2 = pos_full[r.target]
        ax_full.plot([x1, x2], [y1, y2], color="#d62728", lw=1.2 + 1.8 * r.weight, alpha=0.92, zorder=3)
    for c in full_cells:
        x, y = pos_full[c]
        if c in selected_cells:
            ax_full.scatter([x], [y], s=58, color="#d62728", edgecolor="white", linewidth=0.55, zorder=4)
        else:
            ax_full.scatter([x], [y], s=9, color="#4f9bcc", edgecolor="none", alpha=0.82, zorder=2)
    ax_full.set_title(f"Graphe complet : {n_full} stations")
    ax_full.set_aspect("equal")
    ax_full.set_xlim(-1.62, 1.62)
    ax_full.set_ylim(-1.62, 1.62)
    ax_full.axis("off")

    angles = np.linspace(0, 2*np.pi, len(selected_cells), endpoint=False) + np.pi / 6
    pos_zoom = {c: (1.05*np.cos(a), 1.05*np.sin(a)) for c, a in zip(selected_cells, angles)}
    for edge_index, r in enumerate(edge_df.itertuples(index=False)):
        if r.source not in pos_zoom or r.target not in pos_zoom:
            continue
        x1, y1 = pos_zoom[r.source]; x2, y2 = pos_zoom[r.target]
        rad_values = [-0.22, -0.14, -0.08, 0.08, 0.14, 0.22]
        rad = rad_values[edge_index % len(rad_values)]
        patch = FancyArrowPatch((x1, y1), (x2, y2),
                                arrowstyle="-",
                                connectionstyle=f"arc3,rad={rad}",
                                lw=1.6 + 3.2 * r.weight,
                                color="#d62728",
                                alpha=0.88,
                                zorder=1)
        ax_zoom.add_patch(patch)
    for c in selected_cells:
        x, y = pos_zoom[c]
        ax_zoom.scatter([x], [y], s=940, color="#d62728", edgecolor="white", linewidth=2.0, zorder=3)
        ax_zoom.text(x, y, c.replace("B_", "B"), color="white", ha="center", va="center", fontsize=10.5, weight="bold", zorder=4)
    ax_zoom.set_title(f"Sous-graphe retenu : {len(selected_cells)} stations")
    ax_zoom.set_aspect("equal")
    ax_zoom.set_xlim(-1.48, 1.48)
    ax_zoom.set_ylim(-1.40, 1.40)
    ax_zoom.axis("off")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "graph_correlation_bs.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    sample_cells = df["cell_id"].drop_duplicates().head(5).tolist()
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10.5, 6.4), sharex=True, gridspec_kw={"height_ratios": [2.1, 1.2]})
    for c in sample_cells:
        g = df[df.cell_id == c]
        ax1.plot(g.timestamp, g.load, lw=1.1, label=c)
    ax1.set_ylabel("Charge normalisee")
    ax1.set_title("Profils de charge des stations selectionnees")
    ax1.legend(ncol=3, fontsize=8, loc="upper left")
    ax1.grid(True, alpha=0.25)
    energy_mean = df.groupby("timestamp")["energy"].mean()
    ax2.plot(energy_mean.index, energy_mean.values, color="#1f2933", lw=1.8)
    ax2.set_ylabel("Energie moyenne observee")
    ax2.set_xlabel("Temps")
    ax2.set_title("Energie moyenne observee")
    ax2.grid(True, alpha=0.25)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "load_energy_profiles_rl.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(rewards, color="#34568b")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Recompense moyenne")
    ax.set_title("Evolution de la recompense du Safe DQN")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "dqn_training_reward.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    if len(losses) > 0:
        smoothed = pd.Series(losses).rolling(80, min_periods=1).mean()
        ax.plot(smoothed, color="#e45756")
    ax.set_xlabel("Mise a jour")
    ax.set_ylabel("Perte moyenne lisse")
    ax.set_title("Courbe de perte du DQN")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "dqn_loss_curve.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    label_map = {
        "always_on": "Toujours active",
        "static": "Regle statique",
        "static_graph_safe": "Regle statique securisee",
        "dqn": "DQN sans graphe",
        "safe_dqn_graph": "Safe DQN guide par graphe",
    }
    s = summary.copy()
    s["label"] = s["strategy"].map(label_map)
    order = [label_map[k] for k in ["always_on", "static", "static_graph_safe", "dqn", "safe_dqn_graph"]]
    s = s.set_index("label").loc[order].reset_index()

    def bar(metric, filename, title, ylabel):
        fig, ax = plt.subplots(figsize=(10, 5))
        bars = ax.bar(s["label"], s[metric], color=[colors[x] for x in s["label"]], edgecolor="#1f2933", linewidth=0.6)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=18)
        for bar_obj in bars:
            value = bar_obj.get_height()
            ax.text(bar_obj.get_x() + bar_obj.get_width()/2, value, f"{value:.1f}", ha="center", va="bottom", fontsize=8)
        ax.margins(y=0.14)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR / filename, dpi=220, bbox_inches="tight")
        plt.close(fig)

    bar("energy", "energy_comparison_rl.png", "Energie totale estimee par strategie", "Energie")
    bar("energy_gain_pct", "energy_gain_comparison_rl.png", "Gain energetique relatif", "Gain (%)")
    bar("blocking_pct", "blocking_rate_comparison_rl.png", "Taux de blocage estime", "Blocage (%)")
    bar("served_pct", "served_traffic_comparison_rl.png", "Trafic servi", "Service (%)")

    out = evals["safe_dqn_graph"]
    counts = out["action"].map(ACTIONS).value_counts().reindex(ACTIONS.values(), fill_value=0)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(counts.index, counts.values, color=["#4c78a8", "#54a24b", "#e45756"])
    ax.set_ylabel("Nombre de decisions")
    ax.set_title("Distribution des actions du Safe DQN")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "action_distribution_safe_dqn.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    for _, r in s.iterrows():
        ax.scatter(r["blocking_pct"], r["energy_gain_pct"], s=100, color=colors[r["label"]])
        ax.text(r["blocking_pct"] + 0.15, r["energy_gain_pct"], r["label"], fontsize=8)
    ax.set_xlabel("Blocage (%)")
    ax.set_ylabel("Gain energetique (%)")
    ax.set_title("Compromis energie-QoS")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "energy_qos_tradeoff_rl.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 5))
    safe_scen = scen[scen.strategy == "safe_dqn_graph"].copy()
    bars = ax.bar(safe_scen["scenario"], safe_scen["energy_gain_pct"], color="#d62728", edgecolor="#1f2933", linewidth=0.6)
    ax.set_ylabel("Gain energetique (%)")
    ax.set_title("Gain du Safe DQN guide par graphe selon le scenario")
    ax.tick_params(axis="x", rotation=15)
    for bar_obj in bars:
        value = bar_obj.get_height()
        ax.text(bar_obj.get_x() + bar_obj.get_width()/2, value, f"{value:.1f}", ha="center", va="bottom", fontsize=8)
    ax.margins(y=0.14)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "scenario_gain_safe_dqn.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    for f in RESULTS_DIR.glob("*.png"):
        shutil.copy2(f, LATEX_RESULTS_DIR / f.name)


## 9. Analyse de scalabilite

Cette cellule produit les resultats et figures pour les sous-graphes de 6, 16, 32 et 64 stations.


In [9]:
def plot_scalability_results(scale_results: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    colors = {
        "Toujours active": "#d6e9f8",
        "Regle statique": "#9ecae1",
        "Regle statique securisee": "#6baed6",
        "DQN sans graphe": "#2171b5",
        "Safe DQN guide par graphe": "#d62728",
    }
    label_map = {
        "always_on": "Toujours active",
        "static": "Regle statique",
        "static_graph_safe": "Regle statique securisee",
        "dqn": "DQN sans graphe",
        "safe_dqn_graph": "Safe DQN guide par graphe",
    }
    all_rows = []
    safe_rows = []
    for res in scale_results:
        n = res["n_cells"]
        summary = res["summary"].copy()
        summary.insert(0, "n_cells", n)
        summary.insert(1, "n_observations", len(res["df_feat"]))
        summary.insert(2, "n_train", len(res["train_df"]))
        summary.insert(3, "n_test", len(res["test_df"]))
        all_rows.append(summary)
        safe = summary[summary["strategy"] == "safe_dqn_graph"].iloc[0].to_dict()
        safe["n_edges"] = len(res["edge_df"])
        safe["avg_degree"] = 2.0 * len(res["edge_df"]) / max(n, 1)
        safe["last_training_reward"] = float(res["rewards"][-1]) if len(res["rewards"]) else 0.0
        safe_rows.append(safe)

    all_df = pd.concat(all_rows, ignore_index=True)
    safe_df = pd.DataFrame(safe_rows)
    all_df.to_csv(RESULTS_DIR / "scalability_metrics_all_strategies.csv", index=False)
    safe_df.to_csv(RESULTS_DIR / "scalability_metrics_safe_dqn.csv", index=False)

    plot_df = all_df.copy()
    plot_df["label"] = plot_df["strategy"].map(label_map)
    order = [label_map[k] for k in ["static", "static_graph_safe", "dqn", "safe_dqn_graph"]]

    def line_plot(metric, ylabel, title, filename):
        fig, ax = plt.subplots(figsize=(9.5, 5.2))
        for label in order:
            g = plot_df[plot_df["label"] == label].sort_values("n_cells")
            ax.plot(g["n_cells"], g[metric], marker="o", lw=2.3 if "Safe" in label else 1.7,
                    color=colors[label], label=label)
            for _, r in g.iterrows():
                ax.text(r["n_cells"], r[metric], f"{r[metric]:.1f}", fontsize=7.5,
                        ha="center", va="bottom")
        ax.set_xlabel("Nombre de stations du sous-graphe")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_xticks(SCALABILITY_CELLS)
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR / filename, dpi=220, bbox_inches="tight")
        plt.close(fig)

    line_plot("energy_gain_pct", "Gain (%)", "Gain energetique selon la taille du sous-graphe",
              "scalability_energy_gain.png")
    line_plot("blocking_pct", "Blocage (%)", "Taux de blocage selon la taille du sous-graphe",
              "scalability_blocking_rate.png")
    line_plot("served_pct", "Service (%)", "Trafic servi selon la taille du sous-graphe",
              "scalability_served_traffic.png")

    fig, axes = plt.subplots(2, 2, figsize=(11.2, 7.2))
    metrics = [
        ("energy_gain_pct", "Gain energetique (%)"),
        ("blocking_pct", "Blocage (%)"),
        ("served_pct", "Trafic servi (%)"),
        ("switches_per_100", "Commutations / 100 decisions"),
    ]
    s = safe_df.sort_values("n_cells")
    for ax, (metric, ylabel) in zip(axes.ravel(), metrics):
        bars = ax.bar(s["n_cells"].astype(str), s[metric], color="#d62728", edgecolor="#1f2933", linewidth=0.6)
        ax.set_ylabel(ylabel)
        ax.set_xlabel("Stations")
        ax.grid(True, axis="y", alpha=0.22)
        for bar_obj in bars:
            value = bar_obj.get_height()
            ax.text(bar_obj.get_x() + bar_obj.get_width()/2, value, f"{value:.1f}", ha="center", va="bottom", fontsize=8)
    fig.suptitle("Synthese de scalabilite du Safe DQN guide par graphe")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "scalability_safe_dqn_synthesis.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    fig, axes = plt.subplots(2, 2, figsize=(12.5, 10.0))
    for ax, res in zip(axes.ravel(), scale_results):
        n = res["n_cells"]
        corr = res["corr"]
        edge_df = res["edge_df"]
        cells = list(corr.columns)
        angles = np.linspace(0, 2*np.pi, len(cells), endpoint=False)
        radius = 1.0 if n <= 16 else 1.15
        pos = {c: (radius*np.cos(a), radius*np.sin(a)) for c, a in zip(cells, angles)}
        for r in edge_df.itertuples(index=False):
            if r.source not in pos or r.target not in pos:
                continue
            x1, y1 = pos[r.source]; x2, y2 = pos[r.target]
            ax.plot([x1, x2], [y1, y2], color="#d62728", lw=0.65 + 1.55 * r.weight, alpha=0.48, zorder=1)
        node_size = 440 if n <= 16 else (160 if n <= 32 else 80)
        for c in cells:
            x, y = pos[c]
            ax.scatter([x], [y], s=node_size, color="#d62728", edgecolor="white", linewidth=0.75, zorder=3)
        ax.set_title(f"Sous-graphe {n} stations - {len(edge_df)} arcs")
        ax.set_aspect("equal")
        ax.set_xlim(-1.56, 1.56)
        ax.set_ylim(-1.56, 1.56)
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "scalability_subgraphs.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    for f in RESULTS_DIR.glob("*.png"):
        shutil.copy2(f, LATEX_RESULTS_DIR / f.name)
    return all_df, safe_df


## 10. Fonction principale

La fonction `main()` orchestre toute l'experience : donnees, entrainement, evaluation, figures, CSV et analyse de scalabilite.


In [10]:
def create_notebook() -> None:
    print("Notebook is already running in Google Colab.")

def main():
    ensure_dirs()
    raw_df = load_data()
    main_res = run_experiment(raw_df, N_CELLS, safe_episodes=100, raw_episodes=80, train_stride=5, batch_size=32)
    df = main_res["df"]
    corr = main_res["corr"]
    edge_df = main_res["edge_df"]
    full_corr, full_edge_df = build_complete_graph(raw_df, list(corr.columns), edge_df)
    df_feat = main_res["df_feat"]
    train_df = main_res["train_df"]
    test_df = main_res["test_df"]
    rewards = main_res["rewards"]
    losses = main_res["losses"]
    evals = main_res["evals"]
    summary = main_res["summary"]
    scen = scenario_metrics(evals, test_df)
    summary.to_csv(RESULTS_DIR / "summary_metrics_safe_dqn.csv", index=False)
    scen.to_csv(RESULTS_DIR / "scenario_metrics_safe_dqn.csv", index=False)
    corr.to_csv(RESULTS_DIR / "correlation_matrix_bs.csv")
    edge_df.to_csv(RESULTS_DIR / "graph_edges_bs.csv", index=False)
    full_corr.to_csv(RESULTS_DIR / "correlation_matrix_all_bs.csv")
    full_edge_df.to_csv(RESULTS_DIR / "graph_edges_all_bs.csv", index=False)
    pd.DataFrame({"episode": np.arange(1, len(rewards)+1), "avg_reward": rewards}).to_csv(RESULTS_DIR / "training_rewards_safe_dqn.csv", index=False)
    pd.DataFrame({"update": np.arange(1, len(losses)+1), "loss": losses}).to_csv(RESULTS_DIR / "training_loss_safe_dqn.csv", index=False)

    plot_conceptual_diagrams()
    plot_results(df_feat, corr, edge_df, full_corr, full_edge_df, rewards, losses, evals, summary, scen)

    scale_results = []
    for n_cells in SCALABILITY_CELLS:
        random.seed(SEED + n_cells)
        np.random.seed(SEED + n_cells)
        scale_results.append(
            run_experiment(raw_df, n_cells, safe_episodes=50, raw_episodes=35, train_stride=15, batch_size=32)
        )
    scale_all, scale_safe = plot_scalability_results(scale_results)
    create_notebook()

    print("Dataset selectionne:", df_feat.shape[0], "observations,", df_feat.cell_id.nunique(), "stations")
    print("Dataset complet:", raw_df.cell_id.nunique(), "stations")
    print("Periode:", df_feat.timestamp.min(), "->", df_feat.timestamp.max())
    print(summary.to_string(index=False))
    print("\nScalabilite Safe DQN:")
    print(scale_safe[["n_cells", "n_observations", "n_edges", "energy_gain_pct", "served_pct", "blocking_pct", "switches_per_100"]].to_string(index=False))
    print("Resultats:", RESULTS_DIR)
    print("Notebook source: current Google Colab notebook")


## 11. Lancer toute l'experience

Executez cette cellule pour lancer le pipeline complet. Les resultats seront sauvegardes dans Google Drive.


In [11]:
results = main()


Notebook is already running in Google Colab.
Dataset selectionne: 745 observations, 6 stations
Dataset complet: 923 stations
Periode: 2023-01-01 01:00:00 -> 2023-01-08 00:00:00
         strategy      energy  energy_gain_pct  served_pct  blocking_pct  switches_per_100  invalid_actions_pct  avg_reward
        always_on 8403.288490         0.000000  100.000000      0.000000          0.000000                  0.0   -0.002807
           static 5849.216741        30.393717   98.395743      1.604257         30.481283                  0.0    0.262886
static_graph_safe 6475.602392        22.939663  100.000000      0.000000         23.529412                  0.0    0.289814
              dqn 4528.139013        46.114679   95.992282      4.007718         24.064171                  0.0    0.374989
   safe_dqn_graph 5425.183856        35.439752   98.228940      1.771060         23.529412                  0.0    0.367992

Scalabilite Safe DQN:
 n_cells  n_observations  n_edges  energy_gain_pct  serv

## 12. Emplacement des sorties

Apres execution, les fichiers se trouvent dans Google Drive :

- `MyDrive/Implementation_Lahibe/donnees_5g_energy_itu_zindi` : dataset CSV ;
- `MyDrive/Implementation_Lahibe/results_safe_dqn_graphe` : figures et tableaux de resultats ;
- `MyDrive/Implementation_Lahibe/exports_for_latex` : copies organisees pour insertion dans le memoire LaTeX.
